In [3]:
from pyspark.sql.functions import col, to_timestamp

base_path = "abfss://ecommerce@synapseeccomstorage.dfs.core.windows.net"

customers = spark.read.option("header", True).csv(f"{base_path}/raw/customers/*.csv")
orders = spark.read.option("header", True).csv(f"{base_path}/raw/orders/*.csv")
order_items = spark.read.option("header", True).csv(f"{base_path}/raw/order_items/*.csv")
payments = spark.read.option("header", True).csv(f"{base_path}/raw/order_payments/*.csv")
products = spark.read.option("header", True).csv(f"{base_path}/raw/products/*.csv")

customers_clean = customers.dropDuplicates(["customer_id"])

orders_clean = (
    orders
    .dropDuplicates(["order_id"])
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp"))
)

order_items_clean = (
    order_items
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))
)

payments_clean = (
    payments
    .withColumn("payment_sequential", col("payment_sequential").cast("int"))
    .withColumn("payment_installments", col("payment_installments").cast("int"))
    .withColumn("payment_value", col("payment_value").cast("double"))
)

products_clean = products.dropDuplicates(["product_id"])

customers_clean.write.mode("overwrite").parquet(f"{base_path}/silver/customers")
orders_clean.write.mode("overwrite").parquet(f"{base_path}/silver/orders")
order_items_clean.write.mode("overwrite").parquet(f"{base_path}/silver/order_items")
payments_clean.write.mode("overwrite").parquet(f"{base_path}/silver/order_payments")
products_clean.write.mode("overwrite").parquet(f"{base_path}/silver/products")

StatementMeta(sparkpool1, 3, 2, Finished, Available, Finished, False)